# FIBA Tile Montage (MATLAB-modeled) — walkthrough\n
\n
This notebook documents (and partially re-implements in Python) the processing pipeline used by the Fiji/ImageJ plugin **FIBA Tile Montage (MATLAB)**.\n
\n
It’s meant for troubleshooting and clarity: each step is a separate cell so you can see intermediate outputs (images/plots), and the math behind the key steps is written out.\n
\n
The Java plugin is the source of truth; the Python pieces here are a transparency tool to visualize what each stage is doing.

## Pipeline overview\n
\n
For each tile (square crop), the Java pipeline does:\n
\n
1. Contrast stretch (MATLAB `imadjust`-like).\n
2. Apply a separable 2D Tukey window (edge blending).\n
3. Compute 2D FFT, take magnitude, then `fftshift`.\n
4. Build an orientation signal $\text{SOL}(\theta)$ by sampling in polar coordinates and summing along radius.\n
5. Find a statistically significant peak band and estimate fiber angle.\n
6. Build a band-limited frequency mask (radial Tukey × angular Tukey), inverse FFT reconstruct, threshold.\n
\n
Outputs (typical): montage, per-tile panels (`*_crop.jpg`, `*_fft.jpg`, `*_fft.tif`, `*_polar.jpg`, `*_rec.jpg`, `*_mask.jpg`), SOL plots, CSV.

## Key equations and concepts\n
\n
### Tukey window (MATLAB `tukeywin`)\n
Let $n$ be the length and $\alpha \in [0,1]$. Define $t = \frac{i}{n-1}$ for $i=0,\dots,n-1$.\n
\n
$$\n
w(t)=\begin{cases}\n
\tfrac12\left(1+\cos\left(\pi\left(\tfrac{2t}{\alpha}-1\right)\right)\right), & 0\le t < \tfrac{\alpha}{2}\\\n
1, & \tfrac{\alpha}{2} \le t \le 1-\tfrac{\alpha}{2}\\\n
\tfrac12\left(1+\cos\left(\pi\left(\tfrac{2t}{\alpha}-\tfrac{2}{\alpha}+1\right)\right)\right), & 1-\tfrac{\alpha}{2} < t \le 1\n
\end{cases}\n
$$\n
with limiting behaviors: $\alpha=0$ gives a rectangular window and $\alpha=1$ gives a Hann window.\n
\n
The plugin applies this separably as a 2D window $S_{r,c} = w_r w_c$.\n
\n
### Discrete Fourier Transform (DFT) and the FFT\n
For an $n\times n$ image $S[x,y]$ (after normalization + windowing), the 2D DFT is\n
$$\n
K[u,v] = \sum_{x=0}^{n-1} \sum_{y=0}^{n-1} S[x,y] \exp\left(-j2\pi\left(\frac{ux}{n} + \frac{vy}{n}\right)\right)\n
$$\n
for indices $u,v \in \{0,\dots,n-1\}$.\n
\n
The **FFT** (Fast Fourier Transform) is an algorithm that computes these same DFT values exactly, but with much lower computational cost: from $O(N^2)$ to $O(N\log N)$ in 1D (and typically $O(n^2\log n)$ for a 2D $n\times n$ image via separable FFTs).\n
\n
In modeling terms, the DFT/FFT is a discrete transform on samples; it only ‘approximates’ a continuous Fourier transform to the extent that we treat $S[x,y]$ as samples of an underlying continuous image.\n
\n
### FFT magnitude and display scaling\n
Given complex FFT $K(u,v)$, the magnitude is $|K|=\sqrt{\Re(K)^2+\Im(K)^2}$. The plugin uses `fftshift` so DC is centered.\n
\n
The **power spectrum** is\n
$$P(u,v) = |K(u,v)|^2$$\n
and for visualization a robust log-power display is used:\n
$$D(u,v) = \log(1 + P(u,v))$$\n
then a percentile-based contrast stretch (rather than max-normalization) so a single spike does not dominate.\n
\n
### Orientation signal (SOL)\n
With center at $(w,w)$ where $w=n/2$ and $\theta$ measured from the **vertical axis** (row direction):\n
$$\n
B(\theta)=\sum_{r=r_{min}}^{r_{max}} |K|\big(w + r\cos\theta,\; w + r\sin\theta\big)\n
$$\n
using bilinear sampling. $B$ is then mapped into 180 bins and normalized to form $\text{SOL}(\theta)$ with $\sum \text{SOL}=1$.

In [ ]:
# Imports (purely for visualization of plugin outputs + a small conceptual re-implementation)

from __future__ import annotations



from pathlib import Path

import re



import numpy as np



# Pillow is easiest for JPEGs; fall back to matplotlib if Pillow isn't available.

try:

    from PIL import Image

    _HAS_PIL = True

except Exception as e:

    Image = None

    _HAS_PIL = False

    print('WARNING: Pillow not available:', e)



import matplotlib.pyplot as plt



plt.rcParams['figure.figsize'] = (7, 7)

plt.rcParams['image.cmap'] = 'gray'

print('OK (numpy/matplotlib)', '; pillow:', _HAS_PIL)


In [ ]:
# Configure where Fiji/Java outputs are located

#

# FIBA_Tile_Montage defaults to writing into your Downloads folder unless you override `outputDir=...`.

# Files you should see include:

#   <base>_tile_boxes.jpg

#   <base>_tile_montage.jpg

#   <base>_tile_profile.jpg

#   <base>_tile_results.csv

# and per-tile panels:

#   <base>_tile1_crop.jpg, <base>_tile1_fft.jpg, <base>_tile1_mask.jpg, <base>_tile1_polar.jpg, <base>_tile1_rec.jpg, <base>_tile1_sol.jpg, ...



downloads = Path.home() / 'Downloads'



# If you used a custom output folder in the plugin (outputDir=...), set it here:

out_dir = downloads



def _discover_latest_base(out_dir: Path) -> str | None:

    """Pick the newest <base>_tile_montage.jpg in out_dir and return <base>."""

    cands = list(out_dir.glob('*_tile_montage.jpg'))

    if not cands:

        return None

    newest = max(cands, key=lambda p: p.stat().st_mtime)

    return newest.name.replace('_tile_montage.jpg', '')



base = _discover_latest_base(out_dir)



print('Output dir:', out_dir)

print('Exists:', out_dir.exists())

print('Auto-detected base:', base)

if base is None:

    print('No *_tile_montage.jpg found. If your outputs are elsewhere, set out_dir to that folder.')


In [ ]:
# List the newest output files for the detected base name

if base is None:

    raise RuntimeError('No base detected. Set out_dir to where FIBA_Tile_Montage wrote outputs.')



files = sorted(out_dir.glob(f'{base}_*'), key=lambda p: p.stat().st_mtime, reverse=True)

for p in files[:40]:

    print(p.name)

print('Total matching files:', len(files))


## What the plugin measured (printed analysis)



The plugin writes a CSV with the measured angle per tile. This cell loads that CSV, prints a quick summary (mean/std), then plots the per-tile angles and their distribution.


In [ ]:
import csv



csv_path = out_dir / f'{base}_tile_results.csv'

print('CSV:', csv_path, 'exists:', csv_path.exists())



rows = []

if csv_path.exists():

    with csv_path.open('r', newline='', encoding='utf-8') as f:

        for r in csv.DictReader(f):

            # columns: tile_id,left,top,size,pAng,pAng_adj

            rows.append({

                'tile_id': int(r['tile_id']),

                'left': int(r['left']),

                'top': int(r['top']),

                'size': int(r['size']),

                'pAng': float(r['pAng']),

                'pAng_adj': float(r['pAng_adj']),

            })



if not rows:

    print('No CSV rows found. (Did you run the plugin with saveCsv=true?)')

else:

    rows = sorted(rows, key=lambda r: r['tile_id'])

    tile_ids = np.array([r['tile_id'] for r in rows], dtype=int)

    pang = np.array([r['pAng'] for r in rows], dtype=float)

    pang_adj = np.array([r['pAng_adj'] for r in rows], dtype=float)



    finite = np.isfinite(pang_adj)

    mean = float(np.nanmean(pang_adj[finite])) if np.any(finite) else float('nan')

    std = float(np.nanstd(pang_adj[finite])) if np.any(finite) else float('nan')



    print(f'Tiles: {len(rows)}')

    print(f'pAng_adj mean: {mean:.3f} deg')

    print(f'pAng_adj std : {std:.3f} deg')



    # Print a compact table

    print('\nFirst rows:')

    for r in rows[:10]:

        print(f"tile {r['tile_id']:>2d}  left={r['left']:>4d} top={r['top']:>4d} size={r['size']:>4d}  pAng={r['pAng']:>7.2f}  pAng_adj={r['pAng_adj']:>7.2f}")



    # Plot per-tile angles

    plt.figure(figsize=(12, 4))

    plt.plot(tile_ids, pang_adj, '-o', label='pAng_adj')

    plt.axhline(mean, color='C1', linewidth=2, label=f'mean={mean:.2f}')

    plt.xlabel('tile_id')

    plt.ylabel('angle (deg)')

    plt.title('Orientation estimate per tile (from CSV)')

    plt.grid(True, alpha=0.3)

    plt.legend()

    plt.show()



    # Histogram

    plt.figure(figsize=(8, 3))

    plt.hist(pang_adj[finite], bins=min(30, max(5, len(pang_adj)//2)))

    plt.xlabel('angle (deg)')

    plt.ylabel('count')

    plt.title('Distribution of per-tile angles')

    plt.grid(True, alpha=0.3)

    plt.show()


In [ ]:
# Show the key overview outputs (EXCEPT the montage). We'll display the montage again at the very end.



def _show_image(path: Path, title: str | None = None, figsize=(12, 9)):

    if not path.exists():

        print('Missing:', path)

        return

    if _HAS_PIL:

        img = Image.open(path)

        arr = np.asarray(img)

    else:

        arr = plt.imread(str(path))

    plt.figure(figsize=figsize)

    plt.imshow(arr)

    plt.axis('off')

    plt.title(title or path.name)

    plt.show()



tile_boxes_path = out_dir / f'{base}_tile_boxes.jpg'

profile_path = out_dir / f'{base}_tile_profile.jpg'



_show_image(tile_boxes_path, title='Tile boxes overlay (what gets cropped)')

_show_image(profile_path, title='Orientation vs tile (plugin output)', figsize=(12, 4))


In [ ]:
# Inspect one tile's outputs (intermediate visuals)

tile_id = 1  # change me



paths = {

    'crop': out_dir / f'{base}_tile{tile_id}_crop.jpg',

    'fft_jpg': out_dir / f'{base}_tile{tile_id}_fft.jpg',

    'fft_tif': out_dir / f'{base}_tile{tile_id}_fft.tif',

    'polar': out_dir / f'{base}_tile{tile_id}_polar.jpg',

    'sol_plot': out_dir / f'{base}_tile{tile_id}_sol.jpg',

    'rec': out_dir / f'{base}_tile{tile_id}_rec.jpg',

    'mask': out_dir / f'{base}_tile{tile_id}_mask.jpg',

}



for k, p in paths.items():

    print(f'{k:8s}', '->', 'OK' if p.exists() else 'MISSING', p.name)



panels = ['crop', 'fft_jpg', 'polar', 'sol_plot', 'mask', 'rec']

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes = axes.ravel()

for ax, k in zip(axes, panels):

    p = paths[k]

    if not p.exists():

        ax.set_title(f'missing: {k}')

        ax.axis('off')

        continue

    if _HAS_PIL:

        ax.imshow(Image.open(p))

    else:

        ax.imshow(plt.imread(str(p)))

    ax.set_title(k)

    ax.axis('off')

plt.tight_layout()

plt.show()


### How to interpret the per-tile panels



For a given tile ID, the plugin writes these intermediate panels:



- `*_crop.jpg`: the cropped tile after normalization.

- `*_fft.jpg`: the FFT magnitude display (log-power + robust contrast stretch).

- `*_mask.jpg`: the frequency mask used for reconstruction.

- `*_polar.jpg`: a polar visualization of the SOL with the selected peak-band highlighted.

- `*_sol.jpg`: the SOL curve vs angle (with the selected band in red).

- `*_rec.jpg`: the reconstructed/thresholded image used downstream.



If something looks “off” in the final answer, these are the images that explain *where* it went off.


## Intermediate outputs gallery (the “in-between plots”)



These images are the stepping-stone visuals that explain what the app is doing per tile.



Tip: if you ran lots of tiles, reduce `max_tiles` in the code cell below.


In [ ]:
# Gallery helper: show a grid for a given per-tile output kind

def _tile_ids_from_files(suffix: str) -> list[int]:

    ids = []

    rx = re.compile(rf"^{re.escape(base)}_tile(\d+){re.escape(suffix)}$")

    for p in out_dir.glob(f"{base}_tile*{suffix}"):

        m = rx.match(p.name)

        if m:

            ids.append(int(m.group(1)))

    return sorted(set(ids))



def show_tile_gallery(kind: str, tile_ids: list[int] | None = None, max_tiles: int = 12):

    """kind examples: 'crop', 'fft', 'polar', 'mask', 'rec', 'sol'"""

    suffix = {

        'crop': '_crop.jpg',

        'fft': '_fft.jpg',

        'polar': '_polar.jpg',

        'mask': '_mask.jpg',

        'rec': '_rec.jpg',

        'sol': '_sol.jpg',

    }[kind]



    if tile_ids is None:

        tile_ids = _tile_ids_from_files(suffix)

    tile_ids = tile_ids[:max_tiles]

    if not tile_ids:

        print('No tiles found for kind:', kind, 'suffix:', suffix)

        return



    n = len(tile_ids)

    cols = 4

    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))

    axes = np.asarray(axes).ravel()

    for ax in axes:

        ax.axis('off')



    for i, tid in enumerate(tile_ids):

        p = out_dir / f'{base}_tile{tid}{suffix}'

        ax = axes[i]

        if not p.exists():

            ax.set_title(f'tile {tid}: missing')

            continue

        ax.imshow(Image.open(p) if _HAS_PIL else plt.imread(str(p)))

        ax.set_title(f'tile {tid}')

        ax.axis('off')



    plt.suptitle(f'Gallery: {kind} ({suffix})')

    plt.tight_layout()

    plt.show()



# Examples (edit / rerun):

show_tile_gallery('crop', max_tiles=12)

show_tile_gallery('fft', max_tiles=12)

show_tile_gallery('polar', max_tiles=12)

show_tile_gallery('sol', max_tiles=12)

show_tile_gallery('rec', max_tiles=12)


## Diagnosing FFT display artifacts (JPEG vs lossless TIFF)\n
\n
The plugin can save `*_fft.tif` (32-bit float) when `saveFftTif=true`.\n
This helps determine whether visible `clipping/gaps` are introduced by JPEG compression or 8-bit quantization.\n
\n
Below we plot the **center-column** profile for the JPEG, and optionally for the TIFF.

In [ ]:
def read_grayscale_u8(path: Path) -> np.ndarray:

    """Read a JPEG/PNG and return float32 image in [0,1]."""

    if _HAS_PIL:

        img = Image.open(path).convert('L')

        return np.asarray(img, dtype=np.float32) / 255.0



    # Fallback: matplotlib

    arr = plt.imread(str(path)).astype(np.float32)

    if arr.ndim == 3:

        # RGB[A] -> luminance-ish

        arr = arr[..., :3]

        arr = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]

    # plt.imread may already be 0..1 floats

    if arr.max() > 1.5:

        arr = arr / 255.0

    return np.clip(arr, 0.0, 1.0)



fft_jpg = paths['fft_jpg']

if not fft_jpg.exists():

    raise FileNotFoundError(fft_jpg)



jpg = read_grayscale_u8(fft_jpg)

h, w = jpg.shape

cc = jpg[:, w // 2]

print('FFT JPG shape:', jpg.shape)

print('center-col zeros:', int(np.sum(cc == 0.0)), 'of', cc.size)



plt.figure(figsize=(10, 3))

plt.plot(cc)

plt.title('Center column profile (FFT JPG)')

plt.ylim(-0.02, 1.02)

plt.grid(True, alpha=0.3)

plt.show()


In [ ]:
# Optional: read the float TIFF if tifffile is available.\n
fft_tif = paths['fft_tif']\n
if not fft_tif.exists():\n
    print('No TIFF found:', fft_tif)\n
else:\n
    try:\n
        import tifffile as tiff\n
        tif = tiff.imread(str(fft_tif)).astype(np.float32)\n
        cc_t = tif[:, tif.shape[1] // 2]\n
        print('FFT TIF shape:', tif.shape)\n
        print('center-col exact zeros (float):', int(np.sum(cc_t == 0.0)), 'of', cc_t.size)\n
\n
        # visualize robustly\n
        lo, hi = np.quantile(tif, [0.01, 0.999])\n
        tif_vis = np.clip((tif - lo) / (hi - lo + 1e-12), 0, 1)\n
        plt.figure(figsize=(6, 6))\n
        plt.imshow(tif_vis)\n
        plt.axis('off')\n
        plt.title('FFT TIFF (visualized)')\n
        plt.show()\n
\n
        plt.figure(figsize=(10, 3))\n
        plt.plot(cc_t)\n
        plt.title('Center column profile (FFT TIFF, raw float)')\n
        plt.grid(True, alpha=0.3)\n
        plt.show()\n
    except Exception as e:\n
        print('Could not read TIFF (install tifffile?):', e)

## Minimal Python re-implementation of core steps (for intuition)\n
\n
This section re-implements the *conceptual* steps on the `crop` tile using NumPy: Tukey window, FFT + fftshift magnitude, log-power display scaling, and polar sampling to build a SOL-like curve.\n
\n
Again: the Java plugin is the source of truth; this is a transparency tool.

In [ ]:
def tukeywin(n: int, alpha: float) -> np.ndarray:\n
    if n <= 0: return np.array([], dtype=np.float64)\n
    if n == 1: return np.array([1.0], dtype=np.float64)\n
    a = float(np.clip(alpha, 0.0, 1.0))\n
    if a == 0.0: return np.ones(n, dtype=np.float64)\n
    N = n - 1\n
    t = np.arange(n, dtype=np.float64) / N\n
    w = np.ones(n, dtype=np.float64)\n
    a2 = a / 2.0\n
    left = t < a2\n
    mid = (t >= a2) & (t <= (1.0 - a2))\n
    right = t > (1.0 - a2)\n
    w[left] = 0.5 * (1.0 + np.cos(np.pi * ((2.0 * t[left] / a) - 1.0)))\n
    w[mid] = 1.0\n
    w[right] = 0.5 * (1.0 + np.cos(np.pi * ((2.0 * t[right] / a) - (2.0 / a) + 1.0)))\n
    return w\n
\n
def spectrum_display01(imgF_shift_mag: np.ndarray, lo_q=0.01, hi_q=0.999) -> np.ndarray:\n
    v = np.log1p(np.square(imgF_shift_mag))\n
    v = v.copy()\n
    v[v.shape[0] // 2, v.shape[1] // 2] = np.min(v)\n
    lo, hi = np.quantile(v, [lo_q, hi_q])\n
    out = (v - lo) / (hi - lo + 1e-12)\n
    return np.clip(out, 0, 1)\n
\n
crop_img = read_grayscale_u8(paths['crop'])\n
n = crop_img.shape[0]\n
alpha = 0.4\n
w1 = tukeywin(n, alpha)\n
W = np.outer(w1, w1)\n
\n
j = crop_img\n
j = (j - j.min()) / (j.max() - j.min() + 1e-12)\n
avg = float(j.mean())\n
imgS = W * (j - avg) + avg\n
imgS = imgS / (imgS.max() + 1e-12)\n
\n
K = np.fft.fft2(imgS)\n
amp = np.abs(K)\n
imgF = np.fft.fftshift(amp)\n
imgF_disp = spectrum_display01(imgF)\n
\n
fig, ax = plt.subplots(1, 3, figsize=(14, 4))\n
ax[0].imshow(j); ax[0].set_title('crop (normalized)'); ax[0].axis('off')\n
ax[1].imshow(imgS); ax[1].set_title('after Tukey window (ImgS)'); ax[1].axis('off')\n
ax[2].imshow(imgF_disp); ax[2].set_title('FFT magnitude (display)'); ax[2].axis('off')\n
plt.tight_layout(); plt.show()

In [ ]:
# SOL-like curve via polar sampling (conceptual)\n
def bilinear(img: np.ndarray, r: float, c: float) -> float:\n
    h, w = img.shape\n
    r0 = int(np.floor(r)); r1 = int(np.ceil(r))\n
    c0 = int(np.floor(c)); c1 = int(np.ceil(c))\n
    r0 = int(np.clip(r0, 0, h - 1)); r1 = int(np.clip(r1, 0, h - 1))\n
    c0 = int(np.clip(c0, 0, w - 1)); c1 = int(np.clip(c1, 0, w - 1))\n
    dr = r - r0\n
    dc = c - c0\n
    v00 = img[r0, c0]; v10 = img[r1, c0]\n
    v01 = img[r0, c1]; v11 = img[r1, c1]\n
    return float((1 - dr) * (1 - dc) * v00 + dr * (1 - dc) * v10 + (1 - dr) * dc * v01 + dr * dc * v11)\n
\n
def sol_curve(imgF_shift_mag: np.ndarray, rmin: int, rmax: int) -> np.ndarray:\n
    n = imgF_shift_mag.shape[0]\n
    w = n // 2\n
    out = np.zeros(180, dtype=np.float64)\n
    for theta_deg in range(180):\n
        th = np.deg2rad(theta_deg)\n
        s = 0.0\n
        for r in range(rmin, rmax + 1):\n
            rr = w + r * np.cos(th)\n
            cc = w + r * np.sin(th)\n
            s += bilinear(imgF_shift_mag, rr, cc)\n
        out[theta_deg] = s\n
    out = out / (out.sum() + 1e-12)\n
    return out\n
\n
rmin, rmax = 4, (n // 2 - 2)\n
sol = sol_curve(imgF, rmin, rmax)\n
\n
plt.figure(figsize=(12, 3))\n
plt.plot(sol)\n
plt.title('SOL-like curve (Python)')\n
plt.xlabel('angle (deg)')\n
plt.ylabel('normalized sum')\n
plt.grid(True, alpha=0.3)\n
plt.show()\n
print('peak angle (argmax):', int(np.argmax(sol)))

## Final outputs (montage images at the end)



This is the final “deliverable view”:



- The orientation-vs-tile plot

- The montage panel (crop | FFT | polar/SOL | reconstruction)


In [ ]:
# FIBA Tile Montage Report (FINAL CELL)
# Requested output: original image w/ crop-box overlay, then for EACH tile:
#   crop, FFT, polar, mask, and 1D SOL plot.

from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw


def _pil_resample_lanczos():
    # Pillow compatibility across versions
    return getattr(getattr(Image, "Resampling", Image), "LANCZOS", Image.LANCZOS)


def _open_rgb_or_placeholder(path: Path, label: str, size=(256, 256)) -> Image.Image:
    if path.exists():
        try:
            return Image.open(path).convert("RGB")
        except Exception:
            pass

    img = Image.new("RGB", size, (40, 40, 40))
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, size[0] - 1, size[1] - 1], outline=(120, 120, 120), width=2)
    d.multiline_text((10, 10), f"missing\n{label}", fill=(235, 235, 235))
    return img


def _resize_to_height(img: Image.Image, height: int) -> Image.Image:
    if img.height == height:
        return img
    w = max(1, int(round(img.width * (height / img.height))))
    return img.resize((w, height), resample=_pil_resample_lanczos())


def _find_tile_ids(out_dir: Path, base: str) -> list[int]:
    rx = re.compile(rf"^{re.escape(base)}_tile(\d+)_crop\.jpg$")
    ids: list[int] = []
    for p in out_dir.glob(f"{base}_tile*_crop.jpg"):
        m = rx.match(p.name)
        if m:
            ids.append(int(m.group(1)))
    return sorted(set(ids))


def _make_report_image(out_dir: Path, base: str) -> Image.Image:
    tile_boxes_path = out_dir / f"{base}_tile_boxes.jpg"

    tile_ids = _find_tile_ids(out_dir, base)
    if not tile_ids:
        raise FileNotFoundError(
            f"No tiles found in {out_dir} for base={base}. Expected files like {base}_tile1_crop.jpg"
        )

    cols = [
        ("crop", "_crop.jpg"),
        ("FFT", "_fft.jpg"),
        ("polar", "_polar.jpg"),
        ("mask", "_mask.jpg"),
        ("SOL", "_sol.jpg"),
    ]

    thumb_h = 220
    pad = 10
    label_w = 140
    header_h = 60

    # Load + resize and determine column widths
    rows: list[tuple[int, list[Image.Image]]] = []
    col_widths = [0] * len(cols)

    for tid in tile_ids:
        imgs: list[Image.Image] = []
        for j, (col_name, suf) in enumerate(cols):
            p = out_dir / f"{base}_tile{tid}{suf}"
            img = _open_rgb_or_placeholder(p, f"tile{tid} {col_name}")
            img = _resize_to_height(img, thumb_h)
            imgs.append(img)
            col_widths[j] = max(col_widths[j], img.width)
        rows.append((tid, imgs))

    table_w = label_w + pad + sum(col_widths) + pad * (len(cols) + 1)
    table_h = header_h + pad + len(rows) * (thumb_h + pad) + pad

    table = Image.new("RGB", (table_w, table_h), (255, 255, 255))
    d = ImageDraw.Draw(table)

    d.rectangle([0, 0, table_w, header_h], fill=(245, 245, 245))
    d.text((10, 18), "tile", fill=(0, 0, 0))

    x = label_w + pad
    for (name, _), w in zip(cols, col_widths):
        d.text((x + 6, 18), name, fill=(0, 0, 0))
        x += w + pad

    y = header_h + pad
    for tid, imgs in rows:
        d.text((10, y + 10), f"tile {tid}", fill=(0, 0, 0))
        x = label_w + pad
        for img, w in zip(imgs, col_widths):
            d.rectangle([x - 2, y - 2, x + w + 2, y + thumb_h + 2], outline=(220, 220, 220))
            table.paste(img, (x, y))
            x += w + pad
        y += thumb_h + pad

    overlay = _open_rgb_or_placeholder(tile_boxes_path, "tile boxes overlay", size=(table_w, table_w))
    overlay = overlay.resize(
        (table_w, int(round(overlay.height * (table_w / overlay.width)))),
        resample=_pil_resample_lanczos(),
    )

    title_h = 70
    composite_w = table_w
    composite_h = title_h + overlay.height + pad + table.height

    composite = Image.new("RGB", (composite_w, composite_h), (255, 255, 255))
    dc = ImageDraw.Draw(composite)
    dc.text((10, 10), f"FIBA Tile Montage Report — base={base}  tiles={len(tile_ids)}", fill=(0, 0, 0))
    dc.text((10, 35), "Overlay (crop boxes) then per-tile panels", fill=(60, 60, 60))

    y0 = title_h
    composite.paste(overlay, (0, y0))
    y1 = y0 + overlay.height + pad
    composite.paste(table, (0, y1))

    return composite


report_out_path = out_dir / f"{base}_tile_report.png"

print("Output dir:", out_dir)
print("Base:", base)

report_img = _make_report_image(out_dir, base)
report_img.save(report_out_path)

print("Saved report:", report_out_path)

plt.figure(figsize=(14, 14))
plt.imshow(report_img)
plt.axis("off")
plt.show()
